# LifeStack GRPO — training notebook

**Where the real run happened:** I hit repeated import / wheel mismatch issues on **Kaggle** and **Colab** (same stack, different hosted images — I’d fix one thing and another dependency would break). In the end I trained on **Hugging Face** — Jupyter on the Space plus a terminal session — and that environment was stable enough to finish the run without fighting the runtime.

This notebook is still here so you can **re-run the same commands in Colab** if you want; the install cells and troubleshooting section are what I wished had worked reliably for me. Fair warning: hosted notebooks can be picky with TRL / Unsloth / NumPy combos.

**What produced the submission evidence:** `train_run_v4.log` and the plots in the repo are from **episode-level GRPO** (`--episode-train`) with **`LIFESTACK_NO_UNSLOTH=1`** (plain Hugging Face + PEFT). The training cell below sets that in the subprocess only, so the import check up top can still try Unsloth on Colab.

**Rough order:** clone + deps → optional HF token → sanity imports → optional `train_trl.py` patch → episodic dry-run → full v4 (writes `train_run_v4.log`) → optional v2 / resume → plot cells from the log.

In [ ]:
# 1) GPU sanity check
import subprocess
import sys
import platform

subprocess.run(['nvidia-smi'], check=False)
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# 2) Get the repo into /content/Meta-R2
# If you uploaded this notebook from the repo, this still works: it reuses the folder if present.
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/Meta-R2')
REPO_URL = 'https://github.com/oki-dokii/Meta-R2.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repo already exists: {REPO_DIR}')

os.chdir(REPO_DIR)
subprocess.run(['git', 'pull'], check=False)
print('Working directory:', os.getcwd())

In [ ]:
# 3) Install training dependencies
# Keep this runtime training-only. Installing the full demo/RAG stack first can break
# Colab's NumPy/SciPy/sklearn binary compatibility.
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)

# Stable numeric stack for Python 3.12 Colab. Force reinstall fixes errors like:
# ImportError: cannot import name '_center' from numpy._core.umath
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
    'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2',
], check=True)

# Project packages needed by train_trl.py and reward/memory helpers.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'pydantic', 'openenv-core', 'chromadb', 'sentence-transformers', 'gymnasium',
], check=True)

# GRPO/LoRA training stack.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'unsloth', 'trl', 'transformers', 'peft', 'accelerate', 'datasets',
    'bitsandbytes', 'tensorboard', 'huggingface_hub',
], check=True)

# Re-pin numeric packages after dependency resolution so later installs do not leave mixed wheels.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
    'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2',
], check=True)

print('Install complete. If the next import cell still fails, restart runtime and rerun from cell 1.')

In [ ]:
# 4) Optional Hugging Face token for faster downloads and pushing checkpoints
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('HF_TOKEN loaded from Colab Secrets.')
    else:
        print('No HF_TOKEN found. Training can still run, but downloads may be rate-limited.')
except Exception as exc:
    print('Colab Secrets unavailable:', exc)

In [ ]:
# 5) Verify imports and versions
# If this cell fails immediately after installs, use Runtime > Restart runtime,
# then rerun cells from the top. Colab can keep old binary modules loaded.
import numpy
import scipy
import sklearn
import torch
import transformers
import trl
import peft
import accelerate
import datasets
import bitsandbytes
import unsloth

print('numpy:', numpy.__version__)
print('scipy:', scipy.__version__)
print('sklearn:', sklearn.__version__)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('transformers:', transformers.__version__)
print('trl:', trl.__version__)
print('peft:', peft.__version__)
print('accelerate:', accelerate.__version__)
print('datasets:', datasets.__version__)
print('unsloth import: OK')

In [ ]:
# 6) Apply TRL/PEFT compatibility patch if the cloned script does not have it yet
import subprocess
import sys
from pathlib import Path

train_path = Path('/content/Meta-R2/scripts/train_trl.py')
text = train_path.read_text()
original = text

if '_ensure_trl_model_compat' not in text:
    anchor = "def _tensorboard_available() -> bool:\n    try:\n        import tensorboard  # noqa: F401\n        return True\n    except ImportError:\n        return False\n"
    helper = anchor + "\n\ndef _ensure_trl_model_compat(model):\n    if not hasattr(model, 'warnings_issued'):\n        model.warnings_issued = {}\n    return model\n"
    if anchor not in text:
        raise RuntimeError('Could not find _tensorboard_available anchor in train_trl.py')
    text = text.replace(anchor, helper)

full_anchor = 'model, tokenizer = load_model()\n\n    # ── Determine where to start'
if full_anchor in text:
    text = text.replace(
        full_anchor,
        'model, tokenizer = load_model()\n    model = _ensure_trl_model_compat(model)\n\n    # ── Determine where to start',
    )

dry_anchor = 'model, tokenizer = load_model_for_dry_run()\n\n    dataset = generate_dataset'
if dry_anchor in text:
    text = text.replace(
        dry_anchor,
        'model, tokenizer = load_model_for_dry_run()\n    model = _ensure_trl_model_compat(model)\n\n    dataset = generate_dataset',
    )

if text != original:
    train_path.write_text(text)
    print('Patched train_trl.py for TRL/PEFT warnings_issued compatibility.')
else:
    print('Compatibility patch already present.')

subprocess.run([sys.executable, '-m', 'py_compile', str(train_path)], check=True)

## Training configurations (what each run did)

These mirror the commands used for real runs. On the HF server the repo lived at `~/app/Meta-R2`; in Colab it is `/content/Meta-R2`. Always `git pull` before training if you need the latest `scripts/train_trl.py`.

| Flag / setting | Meaning |
|----------------|--------|
| `LIFESTACK_NO_UNSLOTH=1` | Use the **plain HF + PEFT** model path (recommended on large GPUs and on environments where Unsloth is finicky). See `train_trl.py` early-init. |
| `--episode-train` | **Episode-level** training: GRPO sees multi-action rollouts with **episode-shaped rewards**, not only single-step completions. |
| `--episode-horizon 3` | Cap on **actions per episode** during episodic training. |
| `--episodes-per-stage` | How many **episode prompts** per stage in episodic mode. |
| `--episode-warmup-stages 1` | First stage(s) use **single-step** curriculum before episodic fine-tuning. |
| `--stages` | Number of **curriculum stages**. |
| `--prompts-per-stage` | Prompts per stage (paired with episodic knobs as in the script). |
| `--num-train-epochs 3` | **Passes over each stage’s data** per stage (longer optimization than the default `1`). |
| `--max-prompt-length 4096` | **Token budget** for left-truncated prompts in GRPO. |
| `--push-to-hub` / `--hub-repo-id` | Upload the trained artifact to the given **Hub repo**. |
| `--output-dir` | Checkpoint directory (default `./lifestack_model`; **v2** used `./lifestack_model_v2`). |
| `--resume` | Continue from **last checkpoint** + `curriculum_state.json` (**must** match the same training mode as the saved run). |

### Shell equivalents (reference)

**v4 — main evidence run** (log file is what `plot_training.py` and the evidence cells expect):

```bash
cd ~/app/Meta-R2   # or /content/Meta-R2 in Colab
git pull
LIFESTACK_NO_UNSLOTH=1 python scripts/train_trl.py \
  --episode-train \
  --stages 3 \
  --episodes-per-stage 60 \
  --episode-horizon 3 \
  --episode-warmup-stages 1 \
  --prompts-per-stage 60 \
  --num-train-epochs 3 \
  --max-prompt-length 4096 \
  --push-to-hub \
  --hub-repo-id jdsb06/lifestack-grpo-v4 \
  2>&1 | tee train_run_v4.log
```

**v2 — smaller experiment**, separate checkpoints and Hub repo:

```bash
LIFESTACK_NO_UNSLOTH=1 python scripts/train_trl.py \
  --episode-train \
  --stages 2 \
  --prompts-per-stage 40 \
  --episodes-per-stage 40 \
  --episode-horizon 3 \
  --episode-warmup-stages 1 \
  --output-dir ./lifestack_model_v2 \
  --push-to-hub \
  --hub-repo-id jdsb06/lifestack-grpo-v2
```

**Resume** (example from Colab: **prompt-stage** curriculum into `./lifestack_model` — only use if your checkpoint was produced with the same mode):

```bash
python scripts/train_trl.py \
  --stages 3 \
  --prompts-per-stage 50 \
  --output-dir ./lifestack_model \
  --resume
```

The code cells below automate the **v4** and **v2** episodic runs with `LIFESTACK_NO_UNSLOTH=1`; the **resume** cell matches the Colab snippet for prompt-only continuation.

In [ ]:
# 7) Dry-run — episodic path (validates pipeline; tiny cost vs full run)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/Meta-R2')
env = os.environ.copy()
env['LIFESTACK_NO_UNSLOTH'] = '1'  # match v4 / v2 server configuration

subprocess.run(
    [
        sys.executable,
        str(REPO / 'scripts' / 'train_trl.py'),
        '--episode-train',
        '--dry-run',
    ],
    cwd=str(REPO),
    env=env,
    check=True,
)

In [ ]:
# 8) Full training — submission v4 config (tee → train_run_v4.log for evidence / plot_training.py)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/Meta-R2')
LOG = REPO / 'train_run_v4.log'

env = os.environ.copy()
env['LIFESTACK_NO_UNSLOTH'] = '1'

cmd = [
    sys.executable,
    str(REPO / 'scripts' / 'train_trl.py'),
    '--episode-train',
    '--stages', '3',
    '--episodes-per-stage', '60',
    '--episode-horizon', '3',
    '--episode-warmup-stages', '1',
    '--prompts-per-stage', '60',
    '--num-train-epochs', '3',
    '--max-prompt-length', '4096',
    '--push-to-hub',
    '--hub-repo-id', 'jdsb06/lifestack-grpo-v4',
]

proc = subprocess.Popen(
    cmd,
    cwd=str(REPO),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
with open(LOG, 'w', encoding='utf-8') as lf:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        lf.write(line)
ret = proc.wait()
if ret != 0:
    raise subprocess.CalledProcessError(ret, cmd)

In [ ]:
# 8b) Optional — v2 run (smaller curriculum, separate ./lifestack_model_v2 + Hub repo)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/Meta-R2')
env = os.environ.copy()
env['LIFESTACK_NO_UNSLOTH'] = '1'

subprocess.run(
    [
        sys.executable,
        str(REPO / 'scripts' / 'train_trl.py'),
        '--episode-train',
        '--stages', '2',
        '--prompts-per-stage', '40',
        '--episodes-per-stage', '40',
        '--episode-horizon', '3',
        '--episode-warmup-stages', '1',
        '--output-dir', './lifestack_model_v2',
        '--push-to-hub',
        '--hub-repo-id', 'jdsb06/lifestack-grpo-v2',
    ],
    cwd=str(REPO),
    env=env,
    check=True,
)

In [ ]:
# 9) Resume — Colab example (prompt-stage curriculum into ./lifestack_model)
# Use ONLY if your checkpoint was saved from a matching non-episodic run.
# Do not mix with --episode-train checkpoints unless that is what you trained.
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/Meta-R2')

subprocess.run(
    [
        sys.executable,
        str(REPO / 'scripts' / 'train_trl.py'),
        '--stages', '3',
        '--prompts-per-stage', '50',
        '--output-dir', './lifestack_model',
        '--resume',
    ],
    cwd=str(REPO),
    env=os.environ.copy(),
    check=True,
)

---
## 10 · Training Evidence — Plots from the Real Run

These cells re-generate the reward and loss plots from **`train_run_v4.log`**, produced by the **v4** training cell above (same flags as the HF server run, with `LIFESTACK_NO_UNSLOTH=1` and `--episode-train`).

The log and **`grpo_reward_curve_v4.png`** are committed in the repo so judges can inspect evidence **without** re-training.

> **To reproduce plots from your own v4 run**: execute the dry-run cell (7), then the full v4 cell (8), then return here.  
> **To view committed evidence only**: run the plot cells below; they fall back to the repo’s `train_run_v4.log` after `git clone`.

In [ ]:
# 10-a) Ensure matplotlib is available (already present in Colab, but install just in case)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'], check=True)
print('matplotlib ready')

In [ ]:
# 10-b) Generate all training plots from train_run_v4.log
# The script parses both the reward-step lines AND HF Trainer dict lines.
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/Meta-R2')
LOG_FILE = REPO / 'train_run_v4.log'
OUT_DIR  = REPO / 'plots'

if not LOG_FILE.exists():
    print(f'WARNING: {LOG_FILE} not found. Run the full v4 training cell (section 8) or rely on the repo log after clone.')
    print('Pre-committed plots still work in cell 10-c if `train_run_v4.log` exists in the cloned tree.')
else:
    result = subprocess.run(
        [sys.executable, str(REPO / 'scripts' / 'plot_training.py'),
         '--log', str(LOG_FILE),
         '--out', str(OUT_DIR)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)

In [ ]:
# 10-c) Display training evidence inline
from pathlib import Path
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

REPO = Path('/content/Meta-R2')

# Priority order: freshly generated > pre-committed v4 artefact
candidates = [
    ('Training Summary (4-panel)',       REPO / 'plots' / 'training_summary.png'),
    ('Reward Curve',                     REPO / 'plots' / 'reward_curve.png'),
    ('Reward Components',                REPO / 'plots' / 'reward_components.png'),
    ('Loss Curve',                       REPO / 'plots' / 'loss_curve.png'),
    ('Pre-committed Reward Curve (v4)',  REPO / 'grpo_reward_curve_v4.png'),
]

shown = 0
for title, p in candidates:
    if p.exists():
        print(f'--- {title} ---')
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(mpimg.imread(str(p)))
        ax.axis('off')
        ax.set_title(title, fontsize=12, pad=8)
        plt.tight_layout()
        plt.show()
        shown += 1

if shown == 0:
    print('No plot files found. Run cell 10-b after `git clone` (uses committed log) or train with cell 8 first.')

In [ ]:
# 10-d) Print training statistics summary
import re, json
from pathlib import Path

REPO = Path('/content/Meta-R2')
LOG_FILE = REPO / 'train_run_v4.log'

if not LOG_FILE.exists():
    print('train_run_v4.log not found – run training first.')
else:
    text = LOG_FILE.read_text(errors='replace')

    # Parse reward lines
    STEP_RE = re.compile(
        r'\[step\s+(\d+)\]\s+r0=([+-]?\d+\.\d+).*?r_lt=([+-]?\d+\.\d+)'
    )
    matches = STEP_RE.findall(text)

    if matches:
        import numpy as np
        steps   = [int(m[0])   for m in matches]
        rewards = [float(m[1]) for m in matches]
        ltr     = [float(m[2]) for m in matches]
        n = len(rewards)
        q1, q2 = int(n * 0.1), int(n * 0.9)
        print('=' * 52)
        print(f'  LifeStack GRPO  –  Run v4  Training Summary')
        print('=' * 52)
        print(f'  Reward steps logged : {n:>8,}')
        print(f'  Step range          : {steps[0]:>8,} – {steps[-1]:,}')
        print(f'  Mean r0  (all)      : {np.mean(rewards):>8.4f}')
        print(f'  Mean r0  (first 10%): {np.mean(rewards[:q1 or 1]):>8.4f}')
        print(f'  Mean r0  (last  10%): {np.mean(rewards[q2:]):>8.4f}')
        print(f'  Improvement Δ       : {np.mean(rewards[q2:]) - np.mean(rewards[:q1 or 1]):>+8.4f}')
        print(f'  Mean r_lt (all)     : {np.mean(ltr):>8.4f}')
        print(f'  Min  r0             : {min(rewards):>8.4f}')
        print(f'  Max  r0             : {max(rewards):>8.4f}')
        print('=' * 52)
    else:
        print('No [step N] r0=... lines found. Check the log format.')

    # Also parse HF Trainer logs
    DICT_RE = re.compile(r'\{[^{}]+\}')
    losses = []
    for m in DICT_RE.finditer(text):
        try:
            d = eval(m.group(0), {'__builtins__': {}})
            if isinstance(d, dict) and 'loss' in d:
                losses.append(float(d['loss']))
        except Exception:
            pass
    if losses:
        import numpy as np
        print(f'\n  Trainer loss entries: {len(losses):>8,}')
        print(f'  Loss start (avg 10%): {np.mean(losses[:max(1,len(losses)//10)]):>8.4f}')
        print(f'  Loss end   (avg 10%): {np.mean(losses[-max(1,len(losses)//10):]):>8.4f}')

## If this notebook fights you

That was my experience on Colab/Kaggle more than once. The NumPy `_center` / umath thing: rerun the install cell, **restart the runtime**, start from the top — Colab sometimes keeps old `.so` files after a pip upgrade.

If you’re debugging, this helps:

```bash
nvidia-smi
pip show numpy scipy scikit-learn unsloth trl transformers peft torch bitsandbytes accelerate
```

Usually it’s GPU not enabled, a half-installed Unsloth after a reset, or NumPy/SciPy/sklearn not matching what TRL/PEFT expect. If you need a clean train and hosted notebooks keep breaking, a **Hugging Face** Jupyter + terminal setup is what worked for me.